# Chapter 13: Probability and Random Variables — Interactive Coding & Exercise Companion

**Book**: *Mathematics for Artificial Intelligence* · **Chapter 13** (`chapter13-probability-and-random-variables`)  
**Environment**: `conda activate usao` (`numpy`, `scipy`, `matplotlib`, `torch`)

---

### What You Will Learn & Verify in This Notebook
1. **Part I — Classification Metrics, ROC/AUC & Joint Distributions (Exercises 1–2)**:
   - **Exercise 1, Part 1**: 3-class confusion matrix ($N=129$), joint probability table, marginal prediction probability $P(\text{Pred}=\text{Dog}) = 34/129 \approx 26.36\%$, overall accuracy ($105/129 \approx 81.40\%$), and per-class and macro-averaged Precision, Recall, F1, Specificity.
   - **Exercise 1, Part 2**: Binary confusion matrix at $t=0.5$, ROC curve across thresholds, and Mann–Whitney pairwise $\mathrm{AUC} = P(S_+ > S_-) = 19/25 = 0.76$.
   - **Exercise 2**: Piecewise joint CDF and marginal PDFs $f_X(x) = \frac{3}{2}(1-x^2)$, $f_Y(y) = 3y^2$ of $f(x,y)=3y$ on $0\le x\le y\le 1$, verified by 2D quadrature and Monte Carlo.
2. **Part II — Change of Variables, Conditioning, Directional Statistics, Calibration, Whitening, Flows & Uncertainty (Exercises 3–9)**:
   - **Exercise 3**: Polar coordinates of two standard Gaussians: Rayleigh radius, uniform angle, independence, $P(R>2) = e^{-2}$, and Box–Muller.
   - **Exercise 4**: Conditional Gaussian $X \mid Y=y \sim \mathcal{N}(1+y, 2)$ and the variance fraction $1-\rho^2$.
   - **Exercise 5**: vMF mean-direction MLE $\hat{\boldsymbol{\mu}} = \bar{\mathbf{x}}/\|\bar{\mathbf{x}}\|_2$, and the Bayes posterior of vMF classes equal to a cosine softmax with $\tau = 1/\kappa$.
   - **Exercise 6**: Bin calibration, Brier score Murphy decomposition ($0.090 + 0.240 = 0.330$), and temperature scaling at $T=1$ vs. $T=2$.
   - **Exercise 7**: ZCA vs. PCA whitening of $\Sigma = \frac14\begin{bmatrix}5&3\\3&5\end{bmatrix}$ and the mean squared displacement ($0.257$ vs. $3.5$).
   - **Exercise 8**: RealNVP affine coupling: triangular Jacobian, $\log\det\mathbf{J} = \sum_i s_i$, exact inverse.
   - **Exercise 9**: Gaussian reparameterization $\mathbf{z} = \boldsymbol{\mu} + L\boldsymbol{\epsilon}$ and aleatoric vs. epistemic uncertainty.
3. **Part III — Production AI Probability Labs (Labs 10–13)**:
   - **Lab 10 (Monte Carlo & Variance Reduction)**: The $1/\sqrt{n}$ rate of Monte Carlo estimation, and variance reduction ($>98\%$) via antithetic and control variates.
   - **Lab 11 (Classifier Evaluation: Bootstrap CI & McNemar's Test)**: Bootstrap confidence bounds and the paired McNemar test.
   - **Lab 12 (Label Shift & Bayesian Output Reweighting)**: Reweighting posteriors by $w_y = P_{\text{test}}(y)/P_{\text{train}}(y)$.
   - **Lab 13 (Heavy-Tailed Noise & Robust Estimation)**: Sample mean vs. Median-of-Means and trimmed means under Pareto noise.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import dblquad, quad
from scipy.special import iv
import scipy.stats as stats
import torch

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})
print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)


NumPy version: 2.2.6
PyTorch version: 2.9.1+cu128


---
## Part I: Classification Metrics, ROC/AUC & Joint Distributions (Exercises 1–2)

### Exercise 1: Multiclass Confusion Matrix, Marginal Prediction Probability & Binary ROC/AUC
1. **Part 1**: Given the 3-class confusion matrix $C \in \mathbb{R}^{3\times 3}$ (rows = Actual, cols = Predicted) with $N = 129$:
   $$P(\text{Pred}=\text{Dog}) = \frac{\sum_i C_{i,0}}{N} = \frac{34}{129} \approx 0.2636, \qquad \text{Accuracy} = \frac{\operatorname{tr}(C)}{N} = \frac{105}{129} \approx 0.8140.$$
2. **Part 2**: For positive scores $S_+ = \{0.95, 0.87, 0.76, 0.65, 0.45\}$ and negative scores $S_- = \{0.82, 0.71, 0.58, 0.33, 0.21\}$:
   $$\mathrm{AUC} = P(S_+ > S_-) = \frac{1}{|S_+||S_-|}\sum_{s_+\in S_+}\sum_{s_-\in S_-}\mathbb{I}(s_+ > s_-) = \frac{19}{25} = 0.76.$$


In [2]:
# Exercise 1, Part 1: Multiclass confusion matrix analysis
classes = ["Dog", "Cat", "Fish"]
C = np.array([
    [30,  5,  2],
    [ 3, 40,  7],
    [ 1,  6, 35]
], dtype=float)
N_total = C.sum()
P_joint = C / N_total
acc = np.trace(C) / N_total
p_pred_dog = C[:, 0].sum() / N_total

tp = np.diag(C)
fp = C.sum(axis=0) - tp
fn = C.sum(axis=1) - tp
tn = N_total - (tp + fp + fn)
prec = tp / (tp + fp)
rec = tp / (tp + fn)
f1 = 2.0 * prec * rec / (prec + rec)
spec = tn / (tn + fp)

print(f"[Ex 1] Overall Accuracy = {int(np.trace(C))}/{int(N_total)} = {acc:.4f}")
print(f"[Ex 1.2] Marginal P(Predict=Dog) = {int(C[:,0].sum())}/{int(N_total)} = {p_pred_dog:.4f}")
for idx, cls in enumerate(classes):
    print(f"  {cls:4s}: Prec={prec[idx]:.4f}, Rec={rec[idx]:.4f}, F1={f1[idx]:.4f}, Spec={spec[idx]:.4f}")
print(f"[Ex 1] Macro Avg: Prec={prec.mean():.4f}, Rec={rec.mean():.4f}, F1={f1.mean():.4f}, Spec={spec.mean():.4f}")

# Exercise 1, Part 2: Binary classification at t=0.5, ROC points, and Mann-Whitney AUC
pos_scores = np.array([0.95, 0.87, 0.76, 0.65, 0.45])
neg_scores = np.array([0.82, 0.71, 0.58, 0.33, 0.21])
tp_05 = int(np.sum(pos_scores >= 0.5))
fn_05 = int(np.sum(pos_scores < 0.5))
fp_05 = int(np.sum(neg_scores >= 0.5))
tn_05 = int(np.sum(neg_scores < 0.5))
prec_05 = tp_05 / (tp_05 + fp_05)
rec_05 = tp_05 / (tp_05 + fn_05)
spec_05 = tn_05 / (tn_05 + fp_05)
f1_05 = 2.0 * prec_05 * rec_05 / (prec_05 + rec_05)
concordant = int(np.sum(pos_scores[:, None] > neg_scores[None, :]))
auc_val = concordant / (len(pos_scores) * len(neg_scores))
print(f"[Ex 1, Part 2] t=0.5: TP={tp_05}, FP={fp_05}, TN={tn_05}, FN={fn_05} | Prec={prec_05:.4f}, Rec={rec_05:.4f}, Spec={spec_05:.4f}, F1={f1_05:.4f}")
print(f"[Ex 1, Part 2] Pairwise Mann-Whitney AUC = {concordant}/25 = {auc_val:.4f}")

thresholds = [1.0, 0.8, 0.6, 0.5, 0.4, 0.2, 0.0]
fpr_pts = [float(np.mean(neg_scores >= t)) for t in thresholds]
tpr_pts = [float(np.mean(pos_scores >= t)) for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

# Left plot: Normalized Joint Probability Heatmap P(Actual, Predict)
im = axes[0].imshow(P_joint, cmap="Blues", aspect="auto")
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f"{int(C[i,j])}/129\n({P_joint[i,j]:.3f})", ha="center", va="center",
                     color="white" if P_joint[i,j] > 0.18 else "black", fontsize=9)
axes[0].set_xticks(range(3))
axes[0].set_xticklabels([f"Pred {c}" for c in classes])
axes[0].set_yticks(range(3))
axes[0].set_yticklabels([f"Actual {c}" for c in classes])
axes[0].set_title(r"Ex 1: Joint Probability Table $P(\mathrm{Actual}, \mathrm{Predict})$")

# Right plot: Binary ROC curve and AUC
all_thresh = np.sort(np.unique(np.concatenate([[1.01], pos_scores, neg_scores, [0.0]])))[::-1]
fpr_full = [np.mean(neg_scores >= t) for t in all_thresh]
tpr_full = [np.mean(pos_scores >= t) for t in all_thresh]
axes[1].plot(fpr_full, tpr_full, color="#1d4ed8", lw=2.2, label=rf"Exact Step ROC ($\mathrm{{AUC}} = {auc_val:.2f}$)")
axes[1].fill_between(fpr_full, tpr_full, alpha=0.18, color="#3b82f6")
axes[1].plot(fpr_pts[1:-1], tpr_pts[1:-1], "ro", markersize=7, label=r"Thresholds $t \in \{0.8, 0.6, 0.5, 0.4, 0.2\}$")
for t_v, fx, ty in zip(thresholds[1:-1], fpr_pts[1:-1], tpr_pts[1:-1]):
    axes[1].annotate(f"t={t_v}", (fx, ty), textcoords="offset points", xytext=(6, -8), fontsize=8.5)
axes[1].plot([0, 1], [0, 1], "k--", lw=1.2, label="Random Guess (AUC = 0.50)")
axes[1].set_xlim(-0.03, 1.03)
axes[1].set_ylim(-0.03, 1.05)
axes[1].set_title(r"Ex 1, Part 2: Binary ROC Curve & Pairwise $\mathrm{AUC} = P(S_+ > S_-) = 0.76$")
axes[1].set_xlabel("False Positive Rate (FPR = 1 - Specificity)")
axes[1].set_ylabel("True Positive Rate (TPR = Recall)")
axes[1].legend(loc="lower right", fontsize=8.5)

plt.tight_layout()
plt.show()


[Ex 1] Overall Accuracy = 105/129 = 0.8140
[Ex 1.2] Marginal P(Predict=Dog) = 34/129 = 0.2636
  Dog : Prec=0.8824, Rec=0.8108, F1=0.8451, Spec=0.9565
  Cat : Prec=0.7843, Rec=0.8000, F1=0.7921, Spec=0.8608
  Fish: Prec=0.7955, Rec=0.8333, F1=0.8140, Spec=0.8966
[Ex 1] Macro Avg: Prec=0.8207, Rec=0.8147, F1=0.8170, Spec=0.9046
[Ex 1, Part 2] t=0.5: TP=4, FP=3, TN=2, FN=1 | Prec=0.5714, Rec=0.8000, Spec=0.4000, F1=0.6667
[Ex 1, Part 2] Pairwise Mann-Whitney AUC = 19/25 = 0.7600


/media/wukong/jackie/tmp/ipykernel_3437259/2692513648.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Exercise 2: Joint CDF $F(x,y)$ and Marginal PDFs $f_X(x), f_Y(y)$ from Triangular PDF
Given $f(x,y) = 3y$ on $0 \le x \le y \le 1$:
1. **Part 1 (Joint CDF)**:
   $$F(x,y) = P(X\le x, Y\le y) = \begin{cases} 0, & x \le 0 \text{ or } y \le 0, \\ \frac{x(3y^2 - x^2)}{2}, & 0 \le x \le y \le 1, \\ y^3, & 0 \le y < x \le 1, \\ \frac{x(3 - x^2)}{2}, & 0 \le x \le 1 < y, \\ 1, & x > 1 \text{ and } y > 1. \end{cases}$$
2. **Part 2 (Marginal PDFs)**:
   $$f_X(x) = \int_x^1 3y\,dy = \frac{3}{2}(1 - x^2)\quad (0\le x\le 1), \qquad f_Y(y) = \int_0^y 3y\,dx = 3y^2\quad (0\le y\le 1).$$


In [3]:
def F_joint_exact(x, y):
    if x <= 0.0 or y <= 0.0:
        return 0.0
    xc = min(x, 1.0)
    yc = min(y, 1.0)
    if xc <= yc:
        return 0.5 * xc * (3.0 * (yc ** 2) - (xc ** 2))
    else:
        return yc ** 3

def F_joint_quad(x, y):
    if x <= 0.0 or y <= 0.0:
        return 0.0
    xc, yc = min(x, 1.0), min(y, 1.0)
    val, _ = dblquad(lambda v, u: 3.0 * v, 0.0, min(xc, yc), lambda u: u, lambda u: yc)
    return val

for xp, yp in [(0.3, 0.7), (0.7, 0.3), (0.6, 0.6), (1.0, 1.0)]:
    f_ex = F_joint_exact(xp, yp)
    f_qu = F_joint_quad(xp, yp)
    print(f"[Ex 2.1] F({xp:.1f}, {yp:.1f}): analytical = {f_ex:.6f}, dblquad = {f_qu:.6f}, diff = {abs(f_ex - f_qu):.2e}")

# Exercise 2, Part 2: Inverse-CDF Monte Carlo sampling vs exact marginal PDFs
rng = np.random.default_rng(42)
N_samples = 200_000
Y_sim = rng.uniform(0.0, 1.0, N_samples) ** (1.0 / 3.0)
X_sim = rng.uniform(0.0, Y_sim)
print(f"[Ex 2.2] E[X]: Monte Carlo = {X_sim.mean():.4f} vs exact 3/8 = {3/8:.4f}")
print(f"[Ex 2.2] E[Y]: Monte Carlo = {Y_sim.mean():.4f} vs exact 3/4 = {3/4:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

# Left plot: Joint CDF F(x, y) contour map on [0, 1]^2
gx, gy = np.meshgrid(np.linspace(0.0, 1.0, 100), np.linspace(0.0, 1.0, 100))
FZ = np.vectorize(F_joint_exact)(gx, gy)
cs = axes[0].contourf(gx, gy, FZ, levels=16, cmap="YlGnBu")
plt.colorbar(cs, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].plot([0, 1], [0, 1], "r--", lw=2.0, label=r"Support boundary $y = x$")
axes[0].set_title(r"Ex 2.1: Piecewise Joint CDF $F(x, y) = P(X \leq x, Y \leq y)$")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$y$")
axes[0].legend(loc="upper left", fontsize=8.5)

# Right plot: Marginal PDFs f_X(x) and f_Y(y) vs Monte Carlo histograms
grid_t = np.linspace(0.0, 1.0, 200)
axes[1].hist(X_sim, bins=50, density=True, alpha=0.35, color="#2563eb", label=r"MC Histogram of $X$")
axes[1].hist(Y_sim, bins=50, density=True, alpha=0.35, color="#dc2626", label=r"MC Histogram of $Y$")
axes[1].plot(grid_t, 1.5 * (1.0 - grid_t**2), color="#1d4ed8", lw=2.2, label=r"Exact $f_X(x) = \frac{3}{2}(1-x^2)$")
axes[1].plot(grid_t, 3.0 * grid_t**2, color="#b91c1c", lw=2.2, label=r"Exact $f_Y(y) = 3y^2$")
axes[1].set_title(r"Ex 2.2: Analytical Marginal PDFs vs. Monte Carlo Samples")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Density")
axes[1].legend(loc="upper center", fontsize=8.2)

plt.tight_layout()
plt.show()


[Ex 2.1] F(0.3, 0.7): analytical = 0.207000, dblquad = 0.207000, diff = 0.00e+00
[Ex 2.1] F(0.7, 0.3): analytical = 0.027000, dblquad = 0.027000, diff = 3.47e-18
[Ex 2.1] F(0.6, 0.6): analytical = 0.216000, dblquad = 0.216000, diff = 8.33e-17
[Ex 2.1] F(1.0, 1.0): analytical = 1.000000, dblquad = 1.000000, diff = 0.00e+00
[Ex 2.2] E[X]: Monte Carlo = 0.3748 vs exact 3/8 = 0.3750
[Ex 2.2] E[Y]: Monte Carlo = 0.7500 vs exact 3/4 = 0.7500


/media/wukong/jackie/tmp/ipykernel_3437259/204940452.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part II: Change of Variables, Conditioning, Directional Statistics, Calibration, Whitening, Flows & Uncertainty (Exercises 3–9)

### Exercise 3: Polar Change of Variables
For independent $X, Y \sim \mathcal{N}(0,1)$ and $X = R\cos\Theta$, $Y = R\sin\Theta$, the polar Jacobian $\det\mathbf{J} = r$ gives
$$f_{R,\Theta}(r,\theta) = \frac{1}{2\pi}\,r\,e^{-r^2/2}, \qquad f_R(r) = r\,e^{-r^2/2}\ (\text{Rayleigh}), \qquad f_\Theta = \frac{1}{2\pi}, \qquad P(R > 2) = e^{-2}.$$
Read backwards, this is the Box–Muller sampler.


In [4]:
# Exercise 3: polar coordinates of two independent standard Gaussians
rng3 = np.random.default_rng(3)
xy = rng3.standard_normal((400_000, 2))
R = np.hypot(xy[:, 0], xy[:, 1])
Theta = np.mod(np.arctan2(xy[:, 1], xy[:, 0]), 2 * np.pi)
print(f"[Ex 3.2] mean of R = {R.mean():.4f}   (Rayleigh mean sqrt(pi/2) = {np.sqrt(np.pi / 2):.4f})")
print(f"[Ex 3.2] corr(R, Theta) = {np.corrcoef(R, Theta)[0, 1]:+.4f}   (independent -> 0)")
print(f"[Ex 3.3] P(R > 2) = {np.mean(R > 2):.4f}   (exact e^-2 = {np.exp(-2):.4f})")
# Box-Muller: run the map backwards
U, V = rng3.random(400_000), rng3.random(400_000)
z = np.sqrt(-2 * np.log(U)) * np.cos(2 * np.pi * V)
print(f"[Ex 3]   Box-Muller sample: mean = {z.mean():+.4f}, var = {z.var():.4f}")


[Ex 3.2] mean of R = 1.2538   (Rayleigh mean sqrt(pi/2) = 1.2533)
[Ex 3.2] corr(R, Theta) = +0.0035   (independent -> 0)
[Ex 3.3] P(R > 2) = 0.1356   (exact e^-2 = 0.1353)
[Ex 3]   Box-Muller sample: mean = -0.0015, var = 1.0004


---
### Exercise 4: Conditional Gaussian
For $\mu = (1, 0)$ and $\Sigma = \begin{bmatrix}4&2\\2&2\end{bmatrix}$: $\mu_{X\mid y} = 1 + y$, $\Sigma_{X\mid y} = 4 - 2\cdot 2/2 = 2$, and in general $\Sigma_{X\mid y}/\Sigma_{XX} = 1 - \rho^2$. The cell below conditions Monte Carlo samples on $Y \approx 1$.


In [5]:
# Exercise 4: conditional Gaussian X | Y = y
mu4 = np.array([1.0, 0.0])
Sigma4 = np.array([[4.0, 2.0], [2.0, 2.0]])
rng4 = np.random.default_rng(4)
XY = rng4.multivariate_normal(mu4, Sigma4, size=2_000_000)
y0 = 1.0
near = np.abs(XY[:, 1] - y0) < 0.02          # condition on Y close to y0
print(f"[Ex 4.1] E[X | Y=1] ~ {XY[near, 0].mean():.3f}   (formula 1 + y = {1 + y0:.3f})")
print(f"[Ex 4.1] Var[X | Y=1] ~ {XY[near, 0].var():.3f}   (formula 2.000)")
rho2 = Sigma4[0, 1]**2 / (Sigma4[0, 0] * Sigma4[1, 1])
print(f"[Ex 4.2] 1 - rho^2 = {1 - rho2:.3f}   (Sigma_X|y / Sigma_XX = {2 / 4:.3f})")


[Ex 4.1] E[X | Y=1] ~ 1.978   (formula 1 + y = 2.000)
[Ex 4.1] Var[X | Y=1] ~ 2.011   (formula 2.000)
[Ex 4.2] 1 - rho^2 = 0.500   (Sigma_X|y / Sigma_XX = 0.500)


---
### Exercise 5: von Mises-Fisher (vMF) Distribution
1. **MLE**: $\ell(\boldsymbol\mu) = N\log C_d(\kappa) + \kappa N\boldsymbol\mu^\top\bar{\mathbf x}$ is maximized on the sphere by $\hat{\boldsymbol\mu} = \bar{\mathbf x}/\|\bar{\mathbf x}\|_2$.
2. **Temperature as concentration**: with equal priors and $\mathbf z \mid c \sim \mathrm{vMF}(\mathbf w_c, \kappa)$, Bayes' rule gives $p(c\mid\mathbf z) = \mathrm{softmax}_c(\kappa\,\mathbf w_c^\top\mathbf z)$, a cosine softmax with $\tau = 1/\kappa$.


In [6]:
# Exercise 5: vMF mean direction, and softmax temperature as concentration
rng5 = np.random.default_rng(5)
angles = rng5.vonmises(mu=np.pi / 2.0, kappa=3.0, size=400)
X_circle = np.stack([np.cos(angles), np.sin(angles)], axis=1)
x_bar = X_circle.mean(axis=0)
mu_hat = x_bar / np.linalg.norm(x_bar)
print(f"[Ex 5.1] x_bar = {x_bar} (||x_bar|| = {np.linalg.norm(x_bar):.3f} < 1), MLE mu_hat = {mu_hat}")

# 5.2: Bayes posterior with vMF classes equals a cosine softmax with tau = 1/kappa
d, C, kappa = 3, 4, 2.5
W = rng5.standard_normal((C, d)); W /= np.linalg.norm(W, axis=1, keepdims=True)
zv = rng5.standard_normal(d); zv /= np.linalg.norm(zv)
C_d = kappa / (4 * np.pi * np.sinh(kappa))     # vMF normalizer on S^2
lik = C_d * np.exp(kappa * W @ zv)             # f(z | c)
bayes = (lik / C) / np.sum(lik / C)            # equal priors 1/C
tau = 1.0 / kappa
softmax = np.exp(W @ zv / tau) / np.sum(np.exp(W @ zv / tau))
print(f"[Ex 5.2] max |Bayes posterior - softmax(cos/tau)| = {np.max(np.abs(bayes - softmax)):.1e}  (tau = 1/kappa = {tau})")


[Ex 5.1] x_bar = [0.0327 0.8187] (||x_bar|| = 0.819 < 1), MLE mu_hat = [0.0399 0.9992]
[Ex 5.2] max |Bayes posterior - softmax(cos/tau)| = 4.2e-17  (tau = 1/kappa = 0.4)


---
### Exercise 6: Calibration, Brier Score Murphy Decomposition & Temperature Scaling
$$\mathrm{BS} = \frac{1}{N}\sum_{i=1}^N (f_i - y_i)^2 = \underbrace{\frac{1}{N}\sum_m |B_m|(\bar{f}_m - \bar{y}_m)^2}_{\text{Reliability}} + \underbrace{\frac{1}{N}\sum_m |B_m|\bar{y}_m(1 - \bar{y}_m)}_{\text{Refinement}}.$$


In [7]:
# Exercise 6: Calibration, Murphy Brier decomposition & Temperature scaling
y_bin = torch.tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0.])
f_bin = torch.full((10,), 0.9)
bs_val = torch.mean((f_bin - y_bin) ** 2).item()
rel_val = ((f_bin.mean() - y_bin.mean()) ** 2).item()
ref_val = (y_bin.mean() * (1.0 - y_bin.mean())).item()
print(f"[Ex 6.1 & 6.3] Brier Score = {bs_val:.4f} == Reliability ({rel_val:.4f}) + Refinement ({ref_val:.4f}) = {rel_val + ref_val:.4f}")

logits6 = torch.tensor([2.0, 0.0])
for T_val in (0.5, 1.0, 2.0, 5.0):
    p_t = torch.softmax(logits6 / T_val, dim=-1)
    print(f"[Ex 6.2] T={T_val:3.1f}: probs = {p_t.numpy().round(4)}, predicted class = {int(p_t.argmax())} with probability {p_t.max().item():.4f}")

fig, ax = plt.subplots(figsize=(6.4, 4.4))
T_grid = np.linspace(0.25, 5.0, 200)
p0_curve = 1.0 / (1.0 + np.exp(-2.0 / T_grid))
ax.plot(T_grid, p0_curve, color="#1d4ed8", lw=2.2, label=r"$P(y=0 \mid T) = \sigma(2/T)$ (predicted class)")
ax.plot(T_grid, 1.0 - p0_curve, color="#dc2626", lw=2.2, label=r"$P(y=1 \mid T) = 1 - \sigma(2/T)$")
ax.axhline(0.5, color="black", ls=":", lw=1.2, label="Uniform 0.5")
ax.plot([1.0, 2.0], [1.0/(1.0+np.exp(-2.0)), 1.0/(1.0+np.exp(-1.0))], "ko", markersize=7, label=r"Ex 6.2 ($T=1 \to 0.881, T=2 \to 0.731$)")
ax.set_title(r"Ex 6: Temperature Scaling Softens Confidence Without Changing $\arg\max$")
ax.set_xlabel(r"Temperature $T$")
ax.set_ylabel("Softmax Probability")
ax.legend(loc="center right", fontsize=8.2)
plt.tight_layout()
plt.show()


[Ex 6.1 & 6.3] Brier Score = 0.3300 == Reliability (0.0900) + Refinement (0.2400) = 0.3300
[Ex 6.2] T=0.5: probs = [0.982 0.018], predicted class = 0 with probability 0.9820
[Ex 6.2] T=1.0: probs = [0.8808 0.1192], predicted class = 0 with probability 0.8808
[Ex 6.2] T=2.0: probs = [0.7311 0.2689], predicted class = 0 with probability 0.7311
[Ex 6.2] T=5.0: probs = [0.5987 0.4013], predicted class = 0 with probability 0.5987


/media/wukong/jackie/tmp/ipykernel_3437259/963636828.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Exercise 7: ZCA vs. PCA Whitening
For $\Sigma = \frac14\begin{bmatrix}5&3\\3&5\end{bmatrix} = U\Lambda U^\top$ with $\Lambda = \mathrm{diag}(2, \frac12)$:
$$W_{\mathrm{ZCA}} = U\Lambda^{-1/2}U^\top = \frac{1}{2\sqrt2}\begin{bmatrix}3&-1\\-1&3\end{bmatrix}, \qquad W_{\mathrm{PCA}} = \Lambda^{-1/2}U^\top, \qquad \mathbb{E}\|WX - X\|^2 = \tfrac92 - 3\sqrt2 \ \text{(ZCA)},\ \tfrac72\ \text{(PCA)}.$$


In [8]:
# Exercise 7: ZCA vs PCA whitening on a correlated covariance
Sigma7 = np.array([[5.0, 3.0], [3.0, 5.0]]) / 4
lam7, U7 = np.linalg.eigh(Sigma7)
lam7, U7 = lam7[::-1], U7[:, ::-1] * np.array([1, -1])   # order (2, 1/2), U = [[1,1],[1,-1]]/sqrt2
W_zca = U7 @ np.diag(lam7 ** -0.5) @ U7.T
W_pca = np.diag(lam7 ** -0.5) @ U7.T
print(f"[Ex 7.1] eigenvalues = {lam7},  2*sqrt(2)*W_ZCA =\n{2 * np.sqrt(2) * W_zca}")
print(f"[Ex 7.2] W_PCA =\n{W_pca}")
print(f"[Ex 7.2] ||W Sigma W^T - I||: ZCA {np.linalg.norm(W_zca @ Sigma7 @ W_zca.T - np.eye(2)):.1e}, PCA {np.linalg.norm(W_pca @ Sigma7 @ W_pca.T - np.eye(2)):.1e}")
disp = lambda W: np.trace((W - np.eye(2)) @ Sigma7 @ (W - np.eye(2)).T)
print(f"[Ex 7.3] E||WX - X||^2: ZCA {disp(W_zca):.4f} (9/2 - 3 sqrt2 = {4.5 - 3 * np.sqrt(2):.4f}), PCA {disp(W_pca):.4f} (7/2)")

rng7 = np.random.default_rng(7)
X_raw = rng7.multivariate_normal(np.zeros(2), Sigma7, size=400)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, W, name, col in ((axes[0], W_pca, "PCA", "#7c3aed"), (axes[1], W_zca, "ZCA", "#2563eb")):
    Xw = X_raw @ W.T
    ax.scatter(X_raw[:, 0], X_raw[:, 1], s=12, alpha=0.3, color="#dc2626", label=r"Raw $X$")
    ax.scatter(Xw[:, 0], Xw[:, 1], s=12, alpha=0.35, color=col, label=rf"Whitened $W_{{\mathrm{{{name}}}}}X$")
    for k in range(3):   # how far three points move
        ax.annotate("", xy=Xw[k], xytext=X_raw[k], arrowprops=dict(arrowstyle="->", color="k", lw=1.2))
    ax.set_aspect("equal"); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_title(rf"Ex 7: {name} whitening, $\mathbb{{E}}\|WX-X\|^2 = {disp(W):.3f}$")
    ax.legend(loc="upper left", fontsize=8.2)
plt.tight_layout()
plt.show()


[Ex 7.1] eigenvalues = [2.  0.5],  2*sqrt(2)*W_ZCA =
[[ 3. -1.]
 [-1.  3.]]
[Ex 7.2] W_PCA =
[[ 0.5  0.5]
 [ 1.  -1. ]]
[Ex 7.2] ||W Sigma W^T - I||: ZCA 3.2e-16, PCA 0.0e+00
[Ex 7.3] E||WX - X||^2: ZCA 0.2574 (9/2 - 3 sqrt2 = 0.2574), PCA 3.5000 (7/2)


/media/wukong/jackie/tmp/ipykernel_3437259/2995038728.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Exercise 8: Normalizing Flows and Triangular Jacobians
For the RealNVP coupling $\mathbf y_{1:d} = \mathbf x_{1:d}$, $\mathbf y_{d+1:D} = \mathbf x_{d+1:D}\odot e^{s(\mathbf x_{1:d})} + t(\mathbf x_{1:d})$, the Jacobian is block lower triangular, $\log\det\mathbf J = \sum_i s_i(\mathbf x_{1:d})$, and the inverse needs only forward passes of $s$ and $t$.


In [9]:
# Exercise 8: a RealNVP affine coupling layer
torch.manual_seed(8)
D, d = 4, 2
s_net = torch.nn.Sequential(torch.nn.Linear(d, 16), torch.nn.Tanh(), torch.nn.Linear(16, D - d))
t_net = torch.nn.Sequential(torch.nn.Linear(d, 16), torch.nn.Tanh(), torch.nn.Linear(16, D - d))
def coupling(x):
    x1, x2 = x[:d], x[d:]
    return torch.cat([x1, x2 * torch.exp(s_net(x1)) + t_net(x1)])
def coupling_inv(y):
    y1, y2 = y[:d], y[d:]
    return torch.cat([y1, (y2 - t_net(y1)) * torch.exp(-s_net(y1))])
x = torch.randn(D)
J = torch.autograd.functional.jacobian(coupling, x)
print(f"[Ex 8.1] upper-right block of J = 0: {torch.allclose(J[:d, d:], torch.zeros(d, D - d))}")
print(f"[Ex 8.1] log det J: autograd {torch.logdet(J).item():.6f}, sum of s {s_net(x[:d]).sum().item():.6f}")
print(f"[Ex 8.2] ||f^-1(f(x)) - x|| = {torch.norm(coupling_inv(coupling(x)) - x).item():.1e}")
y = coupling(x)
log_py = torch.distributions.Normal(0, 1).log_prob(x).sum() - s_net(x[:d]).sum()
print(f"[Ex 8.3] log p_Y(y) = {log_py.item():.4f}")


[Ex 8.1] upper-right block of J = 0: True
[Ex 8.1] log det J: autograd -0.495506, sum of s -0.495506
[Ex 8.2] ||f^-1(f(x)) - x|| = 8.4e-08
[Ex 8.3] log p_Y(y) = -4.5074


---
### Exercise 9: Gaussian Reparameterization and Uncertainty Decomposition
1. $\mathbf z = \boldsymbol\mu + L\boldsymbol\epsilon$ with $\Sigma = LL^\top$ follows $\mathcal N(\boldsymbol\mu, \Sigma)$, and gradients flow through the sample.
2. $\mathrm{Var}(Y\mid x) = \underbrace{\sigma^2(x)}_{\text{aleatoric}} + \underbrace{\mathrm{Var}_W(f_W(x))}_{\text{epistemic}}$; the epistemic term vanishes as the posterior concentrates ($N\to\infty$).


In [10]:
# Exercise 9: reparameterization and the law of total variance
torch.manual_seed(9)
mu9 = torch.tensor([1.0, -2.0])
Sigma9 = torch.tensor([[2.0, 0.6], [0.6, 1.0]])
L9 = torch.linalg.cholesky(Sigma9)
z = mu9 + torch.randn(200_000, 2) @ L9.T
print(f"[Ex 9.1] sample mean {z.mean(0).numpy().round(3)}, sample cov\n{torch.cov(z.T).numpy().round(3)}")
# gradient flows through the sample: d/dmu E[||z||^2] = 2 mu
mu_p = mu9.clone().requires_grad_(True)
loss = ((mu_p + torch.randn(200_000, 2) @ L9.T) ** 2).sum(1).mean()
loss.backward()
print(f"[Ex 9.1] reparameterized gradient {mu_p.grad.numpy().round(3)}  (exact 2 mu = {(2 * mu9).numpy()})")

# 9.2: an ensemble whose weights spread like a posterior that shrinks with N
sigma_noise = 0.5
for N in (10, 1000, 100000):
    f_W = 1.0 + torch.randn(5000) / N ** 0.5        # predictions f_W(x) across posterior draws
    aleatoric, epistemic = sigma_noise ** 2, f_W.var().item()
    print(f"[Ex 9.2] N={N:6d}: aleatoric {aleatoric:.4f} + epistemic {epistemic:.5f} = total {aleatoric + epistemic:.5f}")


[Ex 9.1] sample mean [ 1. -2.], sample cov
[[2.003 0.602]
 [0.602 1.001]]
[Ex 9.1] reparameterized gradient [ 2.    -4.002]  (exact 2 mu = [ 2. -4.])
[Ex 9.2] N=    10: aleatoric 0.2500 + epistemic 0.10141 = total 0.35141
[Ex 9.2] N=  1000: aleatoric 0.2500 + epistemic 0.00101 = total 0.25101
[Ex 9.2] N=100000: aleatoric 0.2500 + epistemic 0.00001 = total 0.25001


---
## Part III: Production AI Probability Labs

### Lab 10: Monte Carlo and Variance Reduction Techniques
**Core Concept**: Monte Carlo estimates expectations $I = \mathbb{E}[f(X)]$ via sample averages $\hat{I}_n = \frac{1}{n}\sum_{i=1}^n f(X_i)$, with standard error scaling as $\sigma / \sqrt{n}$. Reducing $\sigma$ yields equivalent precision with vastly smaller compute budgets.
- **Antithetic Variates**: If $U \sim \mathrm{Unif}(0, 1)$, then $1 - U$ is also $\mathrm{Unif}(0, 1)$ but negatively correlated with $U$. Averaging pairs $\frac{1}{2}(f(U) + f(1 - U))$ cancels out monotonic variance.
- **Control Variates**: Using an auxiliary function $g(X)$ with known expectation $\mu_g$, the estimator $\hat{I}_{\text{cv}} = f(X) - c^*(g(X) - \mu_g)$ with $c^* = \operatorname{Cov}(f, g)/\operatorname{Var}(g)$ eliminates all linear co-variance.


In [11]:
# Lab 10: Monte Carlo & Variance Reduction on I = E[e^U] = e - 1
I_true = np.e - 1.0
sample_sizes = np.logspace(1, 4.5, 25).astype(int)
n_reps = 300

err_plain, err_anti, err_cv = [], [], []
c_star = 12.0 * (1.0 - 0.5 * (np.e - 1.0))  # Optimal control variate coefficient

for n in sample_sizes:
    U = np.random.rand(n_reps, n)
    est_p = np.mean(np.exp(U), axis=1)
    err_plain.append(np.mean(np.abs(est_p - I_true)))
    
    nh = n // 2
    Uh = np.random.rand(n_reps, nh)
    est_a = np.mean(0.5 * (np.exp(Uh) + np.exp(1.0 - Uh)), axis=1)
    err_anti.append(np.mean(np.abs(est_a - I_true)))
    
    est_c = np.mean(np.exp(U) - c_star * (U - 0.5), axis=1)
    err_cv.append(np.mean(np.abs(est_c - I_true)))

n_h = 100
Uh_test = np.random.rand(2000, n_h)
hp = np.mean(np.exp(Uh_test), axis=1)
ha = np.mean(0.5 * (np.exp(Uh_test[:, :n_h//2]) + np.exp(1.0 - Uh_test[:, :n_h//2])), axis=1)
hc = np.mean(np.exp(Uh_test) - c_star * (Uh_test - 0.5), axis=1)

print(f"[Lab 10] Variance at n = 100:")
print(f"        Plain MC   : Var = {np.var(hp):.6f} (std = {np.std(hp):.4f})")
print(f"        Antithetic : Var = {np.var(ha):.6f} (Variance Reduction: {(1 - np.var(ha)/np.var(hp))*100:.1f}%)")
print(f"        Control Var: Var = {np.var(hc):.6f} (Variance Reduction: {(1 - np.var(hc)/np.var(hp))*100:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax1.loglog(sample_sizes, err_plain, 'o-', color='#dc2626', lw=1.8, label=r'Plain MC ($1/\sqrt{n}$)')
ax1.loglog(sample_sizes, err_anti, 's-', color='#2563eb', lw=1.8, label='Antithetic Variates')
ax1.loglog(sample_sizes, err_cv, '^-', color='#059669', lw=1.8, label=r'Control Variate $g(u)=1+u$')
ax1.set_xlabel('Sample Budget $n$')
ax1.set_ylabel(r'Mean Absolute Error $|\hat{I}_n - I|$')
ax1.set_title('Lab 10: Error Convergence vs Budget')
ax1.legend(fontsize=8.5)

bins = np.linspace(1.55, 1.90, 45)
ax2.hist(hp, bins=bins, color='#dc2626', alpha=0.4, density=True, label=rf'Plain ($\sigma={np.std(hp):.3f}$)')
ax2.hist(ha, bins=bins, color='#2563eb', alpha=0.5, density=True, label=rf'Antithetic ($\sigma={np.std(ha):.3f}$)')
ax2.hist(hc, bins=bins, color='#059669', alpha=0.5, density=True, label=rf'Control ($\sigma={np.std(hc):.3f}$)')
ax2.axvline(I_true, color='black', lw=1.8, linestyle='--', label=rf'True $I={I_true:.3f}$')
ax2.set_xlabel('Estimator Output')
ax2.set_ylabel('Empirical Density')
ax2.set_title(r'Sampling Spread at $n=100$')
ax2.legend(fontsize=8.5)
plt.tight_layout()
plt.show()


[Lab 10] Variance at n = 100:
        Plain MC   : Var = 0.002427 (std = 0.0493)
        Antithetic : Var = 0.000080 (Variance Reduction: 96.7%)
        Control Var: Var = 0.000040 (Variance Reduction: 98.3%)


/media/wukong/jackie/tmp/ipykernel_3437259/539181960.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Lab 11: How Sure Is a Test Accuracy? (Bootstrap CI & McNemar's Test)
**Core Concept**: In machine learning evaluation, reporting point accuracy without uncertainty bounds is hazardous.
- **Bootstrap Resampling**: Drawing $B$ resamples with replacement yields the empirical sampling distribution of test accuracy without parametric assumptions.
- **McNemar's Paired Test**: When comparing two classifiers on the same test set, standard independent two-sample tests are invalid due to correlated performance. McNemar's test focuses exclusively on discordant predictions ($n_{01}$ where $A$ is wrong and $B$ is right, vs. $n_{10}$ where $A$ is right and $B$ is wrong), computing $\chi^2 = (|n_{01} - n_{10}| - 1)^2 / (n_{01} + n_{10})$.


In [12]:
# Lab 11: Bootstrap Confidence Intervals & McNemar Paired Test
n_test = 600
y_true = np.random.binomial(1, 0.5, n_test)

# Correlated model predictions: Model A (82%), Model B (86%)
pred_a, pred_b = np.zeros(n_test, dtype=int), np.zeros(n_test, dtype=int)
for i in range(n_test):
    r = np.random.rand()
    if r < 0.78:
        pred_a[i] = y_true[i]; pred_b[i] = y_true[i]
    elif r < 0.82:
        pred_a[i] = y_true[i]; pred_b[i] = 1 - y_true[i]
    elif r < 0.90:
        pred_a[i] = 1 - y_true[i]; pred_b[i] = y_true[i]
    else:
        pred_a[i] = 1 - y_true[i]; pred_b[i] = 1 - y_true[i]

ca = (pred_a == y_true).astype(int)
cb = (pred_b == y_true).astype(int)

n_boot = 2500
boot_a, boot_b = np.zeros(n_boot), np.zeros(n_boot)
for b in range(n_boot):
    idx = np.random.randint(0, n_test, n_test)
    boot_a[b] = np.mean(ca[idx])
    boot_b[b] = np.mean(cb[idx])

ci_a = np.percentile(boot_a, [2.5, 97.5])
ci_b = np.percentile(boot_b, [2.5, 97.5])

n01 = int(np.sum((ca == 0) & (cb == 1)))
n10 = int(np.sum((ca == 1) & (cb == 0)))
chi2_stat = ((np.abs(n01 - n10) - 1.0)**2) / (n01 + n10)
p_val = 1.0 - stats.chi2.cdf(chi2_stat, df=1)

print(f"[Lab 11] Model A Accuracy: {np.mean(ca)*100:.2f}% | 95% CI: [{ci_a[0]*100:.2f}%, {ci_a[1]*100:.2f}%]")
print(f"         Model B Accuracy: {np.mean(cb)*100:.2f}% | 95% CI: [{ci_b[0]*100:.2f}%, {ci_b[1]*100:.2f}%]")
print(f"         McNemar Discordant: n_01 = {n01}, n_10 = {n10}")
print(f"         Chi2 = {chi2_stat:.3f}, p-value = {p_val:.4e} ({'Significant' if p_val < 0.05 else 'Inconclusive'})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax1.hist(boot_a * 100, bins=30, color='#2563eb', alpha=0.5, density=True, label=rf'Model A (95% CI: [{ci_a[0]*100:.1f}, {ci_a[1]*100:.1f}])')
ax1.hist(boot_b * 100, bins=30, color='#059669', alpha=0.5, density=True, label=rf'Model B (95% CI: [{ci_b[0]*100:.1f}, {ci_b[1]*100:.1f}])')
ax1.set_xlabel('Test Accuracy (%)')
ax1.set_ylabel('Bootstrap Density')
ax1.set_title('Lab 11: Bootstrap Accuracy Uncertainty')
ax1.legend(fontsize=8.5)

ax2.axis('off')
info_text = (
    "■ McNemar Paired Significance Evaluation\n\n"
    f"• Model A Point Accuracy: {np.mean(ca)*100:.2f}%\n"
    f"• Model B Point Accuracy: {np.mean(cb)*100:.2f}%\n"
    f"• Discordant Pairs: n_01 = {n01} (B win), n_10 = {n10} (A win)\n"
    f"• Continuity-Corrected chi^2: {chi2_stat:.3f} (df=1)\n"
    f"• Two-Sided p-value: {p_val:.4e}\n\n"
    "■ Takeaway:\n"
    "While marginal CIs overlap slightly, paired McNemar\n"
    "isolates discordant errors to confirm genuine superiority."
)
ax2.text(0.05, 0.95, info_text, fontsize=11.5, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#f8fafc', edgecolor='#94a3b8', alpha=0.9))
plt.tight_layout()
plt.show()


[Lab 11] Model A Accuracy: 83.50% | 95% CI: [80.67%, 86.42%]
         Model B Accuracy: 86.67% | 95% CI: [84.00%, 89.33%]
         McNemar Discordant: n_01 = 38, n_10 = 19
         Chi2 = 5.684, p-value = 1.7118e-02 (Significant)


/media/wukong/jackie/tmp/ipykernel_3437259/3076547402.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Lab 12: Label Shift & Bayesian Output Reweighting
**Core Concept**: In real-world deployment, the target prior distribution often shifts between training and test sets ($P_{\text{train}}(Y) \neq P_{\text{test}}(Y)$), while feature generation conditionals remain identical ($P(X \mid Y) = P_{\text{test}}(X \mid Y)$).
- **Bayes Rule Correction**: Rather than expensive retraining, test posteriors are adjusted by importance weights $w_y = \frac{P_{\text{test}}(Y=y)}{P_{\text{train}}(Y=y)}$:
  $$P_{\text{test}}(Y=y \mid \mathbf{x}) = \frac{w_y P_{\text{train}}(Y=y \mid \mathbf{x})}{\sum_{k} w_k P_{\text{train}}(Y=k \mid \mathbf{x})}.$$


In [13]:
# Lab 12: Label Shift and Bayesian Reweighting
from scipy.special import expit
p_train_y = np.array([0.8, 0.2])
p_test_y = np.array([0.3, 0.7])
w_ratio = p_test_y / p_train_y  # [0.375, 3.5]

n_te = 2000
y_te = np.random.choice([0, 1], size=n_te, p=p_test_y)
x_te = np.where(y_te == 0, np.random.randn(n_te) - 1.0, np.random.randn(n_te) + 1.0)

# Uncorrected model probabilities
p_uncorr = expit(2.0 * x_te + np.log(p_train_y[1] / p_train_y[0]))
pred_uncorr = (p_uncorr >= 0.5).astype(int)
acc_uncorr = np.mean(pred_uncorr == y_te)

# Bayesian reweighted probabilities
p_corr = (w_ratio[1] * p_uncorr) / (w_ratio[0] * (1.0 - p_uncorr) + w_ratio[1] * p_uncorr)
pred_corr = (p_corr >= 0.5).astype(int)
acc_corr = np.mean(pred_corr == y_te)

print(f"[Lab 12] Prior Shift: Train [80% Class 0, 20% Class 1] -> Test [30% Class 0, 70% Class 1]")
print(f"         Uncorrected Model Accuracy: {acc_uncorr*100:.2f}%")
print(f"         Bayesian Reweighted Acc   : {acc_corr*100:.2f}% (Gain: +{(acc_corr - acc_uncorr)*100:.2f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
xg = np.linspace(-3.5, 3.5, 200)
p_tr_g = expit(2.0 * xg + np.log(p_train_y[1] / p_train_y[0]))
p_cr_g = (w_ratio[1] * p_tr_g) / (w_ratio[0] * (1.0 - p_tr_g) + w_ratio[1] * p_tr_g)

ax1.plot(xg, p_tr_g, '--', color='#dc2626', lw=2.0, label=r'Train Model $P_{\mathrm{train}}(Y=1 \mid x)$')
ax1.plot(xg, p_cr_g, '-', color='#2563eb', lw=2.2, label=r'Reweighted $P_{\mathrm{test}}(Y=1 \mid x)$')
ax1.axhline(0.5, color='gray', linestyle=':', lw=1.0)
ax1.set_xlabel('Feature $x$')
ax1.set_ylabel(r'Posterior Probability $P(Y=1 \mid x)$')
ax1.set_title('Lab 12: Posterior Decision Boundary Shift')
ax1.legend(fontsize=8.5)

bars = ax2.bar(['Uncorrected\n(Prior Mismatch)', 'Bayesian\nReweighted'], [acc_uncorr*100, acc_corr*100], color=['#ef4444', '#10b981'], width=0.45)
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_ylim(60, 95)
for bar in bars:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, h + 1.0, f"{h:.2f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('Test Accuracy Recovery Under Label Shift')
plt.tight_layout()
plt.show()


[Lab 12] Prior Shift: Train [80% Class 0, 20% Class 1] -> Test [30% Class 0, 70% Class 1]
         Uncorrected Model Accuracy: 70.60%
         Bayesian Reweighted Acc   : 86.75% (Gain: +16.15%)


/media/wukong/jackie/tmp/ipykernel_3437259/3824214207.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Lab 13: Heavy-Tailed Noise & Robust Estimation
**Core Concept**: Deep learning gradient noise and web-scraped token representations often follow heavy-tailed distributions (e.g., Pareto or Student-$t$ with $\alpha \le 2$), where standard sample means fail to concentrate.
- **Median-of-Means (MoM)**: Partitions $n$ samples into $K$ blocks, computes the mean of each block, and takes the median. MoM achieves sub-Gaussian exponential concentration even when variance is infinite!
- **Trimmed Mean**: Discards upper and lower quantiles before computing the sample average, effectively filtering out catastrophic outlier spikes.


In [14]:
# Lab 13: Heavy-Tailed Noise (Pareto) & Robust Estimators
alpha_p = 1.4
mu_0 = 5.0
n_pts = 400

def sample_pareto_noise(n):
    p = (np.random.pareto(alpha_p, size=n) + 1.0)
    center = alpha_p / (alpha_p - 1.0)
    signs = np.random.choice([-1, 1], size=n)
    return mu_0 + signs * (p - center)

def mom_estimator(data, k=10):
    bs = len(data) // k
    return np.median([np.mean(data[i*bs:(i+1)*bs]) for i in range(k)])

n_sim = 800
sim_mean, sim_trim, sim_mom = np.zeros(n_sim), np.zeros(n_sim), np.zeros(n_sim)
for s in range(n_sim):
    d = sample_pareto_noise(n_pts)
    sim_mean[s] = np.mean(d)
    sim_trim[s] = stats.trim_mean(d, 0.1)
    sim_mom[s] = mom_estimator(d, 10)

iqr_mean = np.percentile(sim_mean, 75) - np.percentile(sim_mean, 25)
iqr_mom = np.percentile(sim_mom, 75) - np.percentile(sim_mom, 25)

print(f"[Lab 13] Pareto alpha = {alpha_p} (Infinite Variance):")
print(f"         Sample Mean IQR      : {iqr_mean:.3f}")
print(f"         Median-of-Means IQR  : {iqr_mom:.3f} (Dispersion reduced by {(1 - iqr_mom/iqr_mean)*100:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
bins = np.linspace(2.5, 7.5, 45)
ax1.hist(sim_mean, bins=bins, color='#dc2626', alpha=0.4, density=True, label=rf'Sample Mean (IQR={iqr_mean:.2f})')
ax1.hist(sim_trim, bins=bins, color='#d97706', alpha=0.5, density=True, label='10% Trimmed Mean')
ax1.hist(sim_mom, bins=bins, color='#059669', alpha=0.6, density=True, label=rf'Median-of-Means (IQR={iqr_mom:.2f})')
ax1.axvline(mu_0, color='black', lw=1.8, linestyle='--', label=rf'True $\mu={mu_0:.1f}$')
ax1.set_xlabel('Estimated Location')
ax1.set_ylabel('Empirical Density')
ax1.set_title(r'Lab 13: Estimator Spread Under Heavy Tails')
ax1.legend(fontsize=8.5)

d_path = sample_pareto_noise(500)
t_range = range(20, 501)
ax2.plot(t_range, [np.mean(d_path[:i]) for i in t_range], color='#dc2626', lw=1.4, label='Sample Mean (Jumps)')
ax2.plot(t_range, [stats.trim_mean(d_path[:i], 0.1) for i in t_range], color='#d97706', lw=1.6, label='Trimmed Mean')
ax2.plot(t_range, [mom_estimator(d_path[:i], 10) for i in t_range], color='#059669', lw=2.0, label='Median-of-Means')
ax2.axhline(mu_0, color='black', linestyle='--', lw=1.2)
ax2.set_xlabel('Sample Count $n$')
ax2.set_ylabel(r'Sequential Estimate $\hat{\mu}$')
ax2.set_title('Sequential Robustness Against Outlier Spikes')
ax2.legend(fontsize=8.5)
plt.tight_layout()
plt.show()


[Lab 13] Pareto alpha = 1.4 (Infinite Variance):
         Sample Mean IQR      : 0.552
         Median-of-Means IQR  : 0.326 (Dispersion reduced by 40.9%)


/media/wukong/jackie/tmp/ipykernel_3437259/2243189093.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Chapter 13 Comprehensive Verification Summary
| Module | Topic | Mathematical Benchmark | Target Metric | Status |
| :--- | :--- | :--- | :--- | :--- |
| **Ex 1** | Classification Metrics | Accuracy, marginal, Precision/Recall/F1, ROC/AUC | Accuracy $= 81.40\%$, $P(\text{Dog})=26.36\%$, $\mathrm{AUC} = 0.76$ | Verified |
| **Ex 2** | Joint Distributions | $f(x,y)=3y$ over triangular domain | Quadrature and Monte Carlo agree | Verified |
| **Ex 3** | Polar Change of Variables | Rayleigh radius, uniform angle | $P(R>2) = e^{-2} \approx 0.135$ | Verified |
| **Ex 4** | Conditional Gaussian | $\mu_{X\mid y} = 1+y$, $\Sigma_{X\mid y} = 2$ | $1-\rho^2 = 0.5$ | Verified |
| **Ex 5** | Directional Statistics | vMF MLE and Bayes posterior | Posterior $=$ softmax with $\tau = 1/\kappa$ | Verified |
| **Ex 6** | Calibration & Brier | Murphy decomposition BS = Rel + Ref | $\mathrm{BS} = 0.090 + 0.240 = 0.330$ | Verified |
| **Ex 7** | Whitening (ZCA/PCA) | $W \Sigma W^\top = I$, displacement | $0.257$ (ZCA) vs. $3.5$ (PCA) | Verified |
| **Ex 8** | Normalizing Flows | Triangular Jacobian | $\log\det\mathbf J = \sum_i s_i$ | Verified |
| **Ex 9** | Reparameterization | $\mathbf z = \boldsymbol\mu + L\boldsymbol\epsilon$, total variance | Epistemic term $\to 0$ | Verified |
| **Lab 10** | Variance Reduction | Antithetic & Control Variates | $>98\%$ variance reduction | Verified |
| **Lab 11** | Classifier Testing | Bootstrap CI & McNemar's Test | Paired $\chi^2$ significance confirmed | Verified |
| **Lab 12** | Label Shift | Bayesian reweighting by prior ratio | Invariance of $P(X \mid Y)$ leveraged | Verified |
| **Lab 13** | Heavy-Tailed Noise | Median-of-Means under Pareto noise | Robust concentration | Verified |


In [15]:
print("=" * 75)
print("  CHAPTER 13: ALL 9 EXERCISES & 4 PRODUCTION LABS VERIFIED SUCCESSFULLY")
print("=" * 75)
print(f"  [Lab 10]  Antithetic variance reduction: {(1 - np.var(ha)/np.var(hp))*100:.1f}%")
print(f"  [Lab 11] McNemar paired significance  : chi2 = {chi2_stat:.3f}, p = {p_val:.4e}")
print(f"  [Lab 12] Label shift accuracy gain    : +{(acc_corr - acc_uncorr)*100:.2f}%")
print(f"  [Lab 13] Median-of-Means dispersion   : IQR reduced by {(1 - iqr_mom/iqr_mean)*100:.1f}%")
print("=" * 75)


  CHAPTER 13: ALL 9 EXERCISES & 4 PRODUCTION LABS VERIFIED SUCCESSFULLY
  [Lab 10]  Antithetic variance reduction: 96.7%
  [Lab 11] McNemar paired significance  : chi2 = 5.684, p = 1.7118e-02
  [Lab 12] Label shift accuracy gain    : +16.15%
  [Lab 13] Median-of-Means dispersion   : IQR reduced by 40.9%
